In [1]:
# %pip install -U "tensorflow>=2.15,<2.17" "numpy<2"
# print("설치 후 커널 재시작")

In [2]:
import os
import random
import numpy as np
import pandas as pd
import sys
from itertools import product

from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# 실행 환경 확인(재현성/버전 이슈 점검용)
print("TensorFlow:", tf.__version__)
print("Python:", sys.executable)

TensorFlow: 2.21.0
Python: c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Scripts\python.exe


### 1. 데이터 가져오기

In [3]:
# ADASYN 및 스케일링이 이미 반영된 분할 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

print("train/valid/test shape:")
print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

print("\ncolumns:")
print(train_df.columns.tolist())

# 타깃 컬럼명
target_col = "Risk_Label"

print("\ntrain y distribution:")
print(train_df[target_col].astype(int).value_counts())
print(train_df[target_col].astype(int).value_counts(normalize=True))

train/valid/test shape:
train: (2486, 11)
valid: (1438, 11)
test : (822, 11)

columns:
['KOSPI 200 lagged_1_return(%)', 'NASDAQ_return(%)', 'Brent Crude Oil_return(%)', 'GJR_VaR_5_t1', 'Gold Spot_return(%)', 'KOSDAQ_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_OG', 'KOSPI 200_PPO_Hist', 'VKOSPI_Close', 'Risk_Label']

train y distribution:
Risk_Label
0    1665
1     821
Name: count, dtype: int64
Risk_Label
0    0.669751
1    0.330249
Name: proportion, dtype: float64


### 2. 사전 작업

In [4]:
# -------------------------
# 0. 시드 고정(실험 재현성 확보)
# -------------------------
SEED = 1

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)

# -------------------------
# 1. 입력/타깃 분리
# -------------------------
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train = train_df[target_col].astype(int).copy()

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid = valid_df[target_col].astype(int).copy()

X_test_raw = test_df.drop(columns=[target_col]).copy()
y_test = test_df[target_col].astype(int).copy()

# 전부 수치형인지 확인
non_numeric_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_cols:
    raise ValueError(
        f"ANN 입력에는 수치형만 넣는 게 안전합니다. 비수치형 컬럼: {non_numeric_cols}"
    )

# valid/test 컬럼을 train 컬럼 순서와 일치시켜 추론 오류 방지
X_valid_raw = X_valid_raw[X_train_raw.columns]
X_test_raw = X_test_raw[X_train_raw.columns]

# -------------------------
# 2. 입력 데이터는 이미 스케일링된 상태
# -------------------------
X_train_scaled = X_train_raw.copy()
X_valid_scaled = X_valid_raw.copy()
X_test_scaled = X_test_raw.copy()

# -------------------------
# 3. Keras 입력용 ndarray 변환
# -------------------------
X_train_np = np.asarray(X_train_scaled, dtype=np.float32)
y_train_np = np.asarray(y_train, dtype=np.float32)

X_valid_np = np.asarray(X_valid_scaled, dtype=np.float32)
y_valid_np = np.asarray(y_valid, dtype=np.float32)

X_test_np = np.asarray(X_test_scaled, dtype=np.float32)
y_test_np = np.asarray(y_test, dtype=np.float32)

print("train shape:", X_train_np.shape, y_train_np.shape)
print("train class 비율:")
print(pd.Series(y_train_np.astype(int)).value_counts(normalize=True).sort_index())


train shape: (2486, 10) (2486,)
train class 비율:
0    0.669751
1    0.330249
Name: proportion, dtype: float64


### 3. valid data로 하이퍼파라미터 및 early stopping

In [5]:
# 단일 은닉층 ANN 생성 함수
def build_ann(input_dim, hidden_units=41, activation="relu"):
    return Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_units, activation=activation),
        Dense(1, activation="sigmoid")
    ])

# 평가 함수: cutoff 선택과 최종 평가에서 공통 사용
def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    gmean = float(np.sqrt(rec * specificity))

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": rec,
        "precision": prec,
        "specificity": specificity,
        "gmean": gmean,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

# 탐색 대상: 소스 논문 기준 설정(hidden_units=41, ReLU, lr=5.8e-4)을 포함하고,
# 기본 구조는 단일 은닉층 ANN + Adam + early stopping으로 유지
hidden_units_grid = [32, 41, 64, 128]
activation_grid = ["relu", "tanh"]
lr_grid = [1e-3, 5.8e-4, 5e-4, 1e-4]
optimizer_grid = ["adam"]

# 모든 조합 생성
search_space = [
    {
        "hidden_units": hidden_units,
        "activation": activation,
        "lr": lr,
        "optimizer": optimizer
    }
    for hidden_units, activation, lr, optimizer in product(
        hidden_units_grid,
        activation_grid,
        lr_grid,
        optimizer_grid
    )
]

# 선택 우선순위: valid_gmean > valid_f1 > valid_recall > valid_log_loss
# cutoff는 0.5 고정이 아니라 valid set에서 함께 선택
search_history = []

for i, cfg in enumerate(search_space, start=1):
    tf.keras.backend.clear_session()
    set_seed(SEED)

    model = build_ann(
        input_dim=X_train_np.shape[1],
        hidden_units=cfg["hidden_units"],
        activation=cfg["activation"]
    )

    if cfg["optimizer"] == "adam":
        optimizer = Adam(learning_rate=cfg["lr"])
    else:
        raise ValueError(f"지원하지 않는 optimizer: {cfg['optimizer']}")

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    # 학습 중단은 val_loss 기준(과적합 완화)
    es = EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=20,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        X_train_np,
        y_train_np,
        validation_data=(X_valid_np, y_valid_np),
        epochs=300,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    valid_prob = model.predict(X_valid_np, verbose=0).ravel()
    valid_loss = float(np.min(history.history["val_loss"]))

    # 모델별 확률 분포에 맞춰 cutoff 후보 생성 + 0.5도 후보에 포함
    threshold_grid = np.unique(
        np.r_[
            np.quantile(valid_prob, np.linspace(0.01, 0.99, 99)),
            0.5
        ]
    )
    threshold_grid = threshold_grid[(threshold_grid >= 0) & (threshold_grid <= 1)]

    cutoff_rows = []
    for cutoff in threshold_grid:
        m = compute_binary_metrics(y_valid_np, valid_prob, threshold=float(cutoff))
        cutoff_rows.append({
            "threshold": float(cutoff),
            "accuracy": m["accuracy"],
            "f1": m["f1"],
            "recall": m["recall"],
            "precision": m["precision"],
            "specificity": m["specificity"],
            "gmean": m["gmean"]
        })

    cutoff_df = (
        pd.DataFrame(cutoff_rows)
        .sort_values(
            by=["gmean", "f1", "recall"],
            ascending=[False, False, False]
        )
        .reset_index(drop=True)
    )
    best_cutoff_row = cutoff_df.iloc[0]

    row = {
        **cfg,
        "best_cutoff": float(best_cutoff_row["threshold"]),
        "valid_accuracy": float(best_cutoff_row["accuracy"]),
        "valid_f1": float(best_cutoff_row["f1"]),
        "valid_recall": float(best_cutoff_row["recall"]),
        "valid_precision": float(best_cutoff_row["precision"]),
        "valid_specificity": float(best_cutoff_row["specificity"]),
        "valid_gmean": float(best_cutoff_row["gmean"]),
        "valid_log_loss": valid_loss,
        "epochs_run": len(history.history["loss"])
    }
    search_history.append(row)

    print(
        f"[Config {i:02d}] {cfg} -> "
        f"cutoff={row['best_cutoff']:.6f}, "
        f"valid_gmean={row['valid_gmean']:.6f}, "
        f"valid_f1={row['valid_f1']:.6f}, "
        f"valid_recall={row['valid_recall']:.6f}, "
        f"best_val_loss={row['valid_log_loss']:.6f}"
    )

search_df = (
    pd.DataFrame(search_history)
    .sort_values(
        by=["valid_gmean", "valid_f1", "valid_recall", "valid_log_loss"],
        ascending=[False, False, False, True]
    )
    .reset_index(drop=True)
)

best_row = search_df.iloc[0]
best_config = {
    "hidden_units": int(best_row["hidden_units"]),
    "activation": str(best_row["activation"]),
    "lr": float(best_row["lr"]),
    "optimizer": str(best_row["optimizer"])
}
best_cutoff = float(best_row["best_cutoff"])

print("\n[Top 10 ANN Search Results]")
print(search_df.head(10).to_string(index=False))

print("\n선택된 설정:", best_config)
print(f"선택된 cutoff: {best_cutoff:.6f}")
print(
    f"선택 기준 valid_gmean={best_row['valid_gmean']:.6f}, "
    f"valid_f1={best_row['valid_f1']:.6f}, "
    f"valid_recall={best_row['valid_recall']:.6f}, "
    f"best_val_loss={best_row['valid_log_loss']:.6f}"
)

# 선택된 설정으로 최종 모델 재학습
tf.keras.backend.clear_session()
set_seed(SEED)

ann_model = build_ann(
    input_dim=X_train_np.shape[1],
    hidden_units=best_config["hidden_units"],
    activation=best_config["activation"]
)
ann_model.compile(
    optimizer=Adam(learning_rate=best_config["lr"]),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

history = ann_model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_valid_np, y_valid_np),
    epochs=300,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)



[Config 01] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.001, 'optimizer': 'adam'} -> cutoff=0.291572, valid_gmean=0.685570, valid_f1=0.335950, valid_recall=0.660494, best_val_loss=0.406526
[Config 02] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.00058, 'optimizer': 'adam'} -> cutoff=0.298854, valid_gmean=0.685570, valid_f1=0.335950, valid_recall=0.660494, best_val_loss=0.414045
[Config 03] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.0005, 'optimizer': 'adam'} -> cutoff=0.300387, valid_gmean=0.689146, valid_f1=0.339089, valid_recall=0.666667, best_val_loss=0.415066
[Config 04] {'hidden_units': 32, 'activation': 'relu', 'lr': 0.0001, 'optimizer': 'adam'} -> cutoff=0.316841, valid_gmean=0.685570, valid_f1=0.335950, valid_recall=0.660494, best_val_loss=0.428206
[Config 05] {'hidden_units': 32, 'activation': 'tanh', 'lr': 0.001, 'optimizer': 'adam'} -> cutoff=0.285463, valid_gmean=0.683818, valid_f1=0.331797, valid_recall=0.666667, best_val_loss=0.407903
[Config 06] {'h

### 4. 평가

In [6]:
# -------------------------
# valid / test 예측 및 지표 계산
# -------------------------
valid_prob = ann_model.predict(X_valid_np, verbose=0).ravel()
test_prob = ann_model.predict(X_test_np, verbose=0).ravel()

valid_metrics = compute_binary_metrics(y_valid_np, valid_prob, threshold=best_cutoff)
test_metrics = compute_binary_metrics(y_test_np, test_prob, threshold=best_cutoff)

print(f"\n[Selected Cutoff] {best_cutoff:.6f}")

# 핵심 지표 + 오차행렬 구성요소 출력
print("\n[Validation Metrics]")
for k in ["accuracy", "f1", "recall", "precision", "specificity", "gmean", "tn", "fp", "fn", "tp"]:
    print(f"{k}: {valid_metrics[k]:.6f}" if isinstance(valid_metrics[k], float) else f"{k}: {valid_metrics[k]}")

print("\n[Test Metrics]")
for k in ["accuracy", "f1", "recall", "precision", "specificity", "gmean", "tn", "fp", "fn", "tp"]:
    print(f"{k}: {test_metrics[k]:.6f}" if isinstance(test_metrics[k], float) else f"{k}: {test_metrics[k]}")

# -------------------------
# 결과 확인용 DataFrame 생성
# 저장은 하지 않음
# -------------------------
valid_pred_df = valid_df.copy()
valid_pred_df["ANN_Prob"] = valid_metrics["y_prob"]
valid_pred_df["ANN_Pred"] = valid_metrics["y_pred"]
valid_pred_df["ANN_Cutoff"] = best_cutoff

test_pred_df = test_df.copy()
test_pred_df["ANN_Prob"] = test_metrics["y_prob"]
test_pred_df["ANN_Pred"] = test_metrics["y_pred"]
test_pred_df["ANN_Cutoff"] = best_cutoff

best_config_df = pd.DataFrame([{
    **best_config,
    "best_cutoff": best_cutoff,
    "valid_accuracy": valid_metrics["accuracy"],
    "valid_f1": valid_metrics["f1"],
    "valid_recall": valid_metrics["recall"],
    "valid_precision": valid_metrics["precision"],
    "valid_specificity": valid_metrics["specificity"],
    "valid_gmean": valid_metrics["gmean"],
    "test_accuracy": test_metrics["accuracy"],
    "test_f1": test_metrics["f1"],
    "test_recall": test_metrics["recall"],
    "test_precision": test_metrics["precision"],
    "test_specificity": test_metrics["specificity"],
    "test_gmean": test_metrics["gmean"]
}])

print("\n[Best Config]")
display(best_config_df)

print("\n[Validation Prediction Preview]")
display(valid_pred_df.head())

print("\n[Test Prediction Preview]")
display(test_pred_df.head())


[Selected Cutoff] 0.300387

[Validation Metrics]
accuracy: 0.707232
f1: 0.339089
recall: 0.666667
precision: 0.227368
specificity: 0.712382
gmean: 0.689146
tn: 909
fp: 367
fn: 54
tp: 108

[Test Metrics]
accuracy: 0.605839
f1: 0.276786
recall: 0.681319
precision: 0.173669
specificity: 0.596443
gmean: 0.637470
tn: 436
fp: 295
fn: 29
tp: 62

[Best Config]


,hidden_units,activation,lr,optimizer,best_cutoff,valid_accuracy,valid_f1,valid_recall,valid_precision,valid_specificity,valid_gmean,test_accuracy,test_f1,test_recall,test_precision,test_specificity,test_gmean
0,32,relu,0.0005,adam,0.300387,0.707232,0.339089,0.666667,0.227368,0.712382,0.689146,0.605839,0.276786,0.681319,0.173669,0.596443,0.63747



[Validation Prediction Preview]


,KOSPI 200 lagged_1_return(%),NASDAQ_return(%),Brent Crude Oil_return(%),GJR_VaR_5_t1,Gold Spot_return(%),KOSDAQ_return(%),KOSPI 200_BB_width,KOSPI 200_OG,KOSPI 200_PPO_Hist,VKOSPI_Close,Risk_Label,ANN_Prob,ANN_Pred,ANN_Cutoff
0,0.512922,0.608325,0.390108,0.894685,0.513998,0.569199,0.066433,0.581208,0.681697,0.057644,0,0.214646,0,0.300387
1,0.575784,0.535391,0.636761,0.902472,0.627651,0.469253,0.060560,0.583026,0.691211,0.064411,0,0.249161,0,0.300387
2,0.591664,0.597625,0.373687,0.909866,0.554072,0.547612,0.049395,0.574811,0.696083,0.068922,0,0.237156,0,0.300387
3,0.566303,0.495450,0.914220,0.916964,0.483441,0.517608,0.051940,0.587097,0.708737,0.064912,0,0.220911,0,0.300387
4,0.613469,0.469315,0.544458,0.923751,0.547913,0.494469,0.046719,0.600154,0.716541,0.068421,0,0.332478,1,0.300387



[Test Prediction Preview]


,KOSPI 200 lagged_1_return(%),NASDAQ_return(%),Brent Crude Oil_return(%),GJR_VaR_5_t1,Gold Spot_return(%),KOSDAQ_return(%),KOSPI 200_BB_width,KOSPI 200_OG,KOSPI 200_PPO_Hist,VKOSPI_Close,Risk_Label,ANN_Prob,ANN_Pred,ANN_Cutoff
0,0.770835,0.855129,0.544847,0.745872,0.620760,0.550300,0.150466,0.435893,0.726038,0.355639,0,0.111489,0,0.300387
1,0.574190,0.654016,0.468905,0.761492,0.539233,0.648210,0.124471,0.723733,0.779706,0.317043,0,0.215983,0,0.300387
2,0.685769,0.511227,0.659218,0.768221,0.491387,0.460869,0.128166,0.566846,0.797461,0.325815,0,0.335274,1,0.300387
3,0.500807,0.530720,0.543905,0.770449,0.578883,0.428834,0.125817,0.496463,0.788805,0.346617,0,0.407548,1,0.300387
4,0.489033,0.766135,0.597926,0.787416,0.638283,0.464544,0.116975,0.536817,0.781796,0.344110,0,0.151297,0,0.300387
